# Probability demos

Each claim in the probability chapter can be checked with arithmetic: count
through a population, enumerate every possible outcome, or simulate a large
number of draws. This notebook does that for Bayes rule, maximum likelihood,
the $\sigma/\sqrt{n}$ error bar, the central limit theorem, and Bessel's
correction, and prints the same numbers the chapter quotes so the two can be
compared line by line.

Matching sections of [04_probability.md](04_probability.md):

- §3.3 Bayes rule, from counting: the rare-disease test on 10,000 people (17%).
- §3.4 Maximum likelihood: the 7-heads-in-10 table and $\hat p = h/n$.
- §3.1 and §3.6 Averages: the die's mean 3.5 and standard deviation 1.71, and the $\sigma/\sqrt{n}$ error bar.
- §3.6 The bell shape: the central limit theorem on uniform and exponential draws.
- §3.7 Bessel's correction: all 216 samples of three die rolls (1.94 vs 2.92).

numpy only, fixed seeds, runs in a few seconds on a CPU.

In [1]:
import itertools

import numpy as np

## 1. Bayes rule with counts (§3.3)

The disease affects 1% of people, the test flags 99% of sick people
(sensitivity), and it clears 95% of healthy people (specificity), so 5% of
healthy people get a false positive. Out of 10,000 people, 99 sick people and
495 healthy people test positive, so a positive result means a
$99/594 \approx 17\%$ chance of being sick. The cell builds the count table,
computes the same posterior with the formula $P(+ \mid \text{sick})P(\text{sick})/P(+)$,
repeats it for a symptomatic group with a 10% prior (69%), and confirms the
17% with a simulation of 1,000,000 patients.

In [2]:
prevalence = 0.01    # P(sick), the prior
sensitivity = 0.99   # P(+ | sick)
specificity = 0.95   # P(- | healthy)
population = 10_000

sick = round(population * prevalence)
healthy = population - sick
true_pos = round(sick * sensitivity)
false_neg = sick - true_pos
false_pos = round(healthy * (1 - specificity))
true_neg = healthy - false_pos

print(f"Out of {population:,} people:\n")
print(f"{'':10s}{'test +':>10s}{'test -':>10s}{'total':>10s}")
print(f"{'sick':10s}{true_pos:>10,d}{false_neg:>10,d}{sick:>10,d}")
print(f"{'healthy':10s}{false_pos:>10,d}{true_neg:>10,d}{healthy:>10,d}")
print(f"{'total':10s}{true_pos + false_pos:>10,d}{false_neg + true_neg:>10,d}{population:>10,d}\n")

posterior_counts = true_pos / (true_pos + false_pos)
print(f"P(sick | +) from counts  = {true_pos} / {true_pos + false_pos} = {posterior_counts:.4f}")

p_pos = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
posterior_formula = sensitivity * prevalence / p_pos
print(f"P(sick | +) from formula = {sensitivity * prevalence:.4f} / {p_pos:.4f} = {posterior_formula:.4f}")
assert (true_pos, false_pos) == (99, 495)
assert abs(posterior_counts - posterior_formula) < 1e-12

prior_symptomatic = 0.10
post_symptomatic = sensitivity * prior_symptomatic / (
    sensitivity * prior_symptomatic + (1 - specificity) * (1 - prior_symptomatic))
print(f"\nSame test, 10% prior (people with symptoms): P(sick | +) = {post_symptomatic:.2f}")

rng = np.random.default_rng(42)
n_sim = 1_000_000
is_sick = rng.random(n_sim) < prevalence
tests_pos = np.where(is_sick, rng.random(n_sim) < sensitivity,
                     rng.random(n_sim) < (1 - specificity))
mc = is_sick[tests_pos].mean()
print(f"Simulation of {n_sim:,} patients: P(sick | +) = {mc:.4f}")
assert abs(mc - posterior_formula) < 0.01

Out of 10,000 people:

              test +    test -     total
sick              99         1       100
healthy          495     9,405     9,900
total            594     9,406    10,000

P(sick | +) from counts  = 99 / 594 = 0.1667
P(sick | +) from formula = 0.0099 / 0.0594 = 0.1667

Same test, 10% prior (people with symptoms): P(sick | +) = 0.69
Simulation of 1,000,000 patients: P(sick | +) = 0.1668


Of the 594 positives, 495 come from the healthy group, because the healthy
group is 99 times larger than the sick group. Precision under class imbalance
is the same calculation: the fraction of flagged cases that are real depends
on how rare the real cases are, not only on the classifier's error rates.

## 2. Maximum likelihood on a coin (§3.4)

For a coin with heads probability $p$, the probability of a particular
sequence with $h$ heads in $n$ flips is $p^h(1-p)^{n-h}$. The cell reproduces
the chapter's table for 7 heads in 10 flips (0.00098, 0.0018, 0.0022, 0.0017
at $p$ = 0.5, 0.6, 0.7, 0.8) and finds the best $p$ on a fine grid. Expected:
the maximum is at $p = 0.7 = h/n$.

In [3]:
h, n = 7, 10
for p in [0.5, 0.6, 0.7, 0.8]:
    print(f"p = {p}:  p^{h} (1-p)^{n - h} = {p ** h * (1 - p) ** (n - h):.2g}")

grid = np.linspace(0.001, 0.999, 999)
lik = grid ** h * (1 - grid) ** (n - h)
p_best = grid[np.argmax(lik)]
print(f"\nbest p on a grid of 999 values: {p_best:.3f}  (h/n = {h / n})")
assert abs(p_best - h / n) < 1e-3

p = 0.5:  p^7 (1-p)^3 = 0.00098
p = 0.6:  p^7 (1-p)^3 = 0.0018
p = 0.7:  p^7 (1-p)^3 = 0.0022
p = 0.8:  p^7 (1-p)^3 = 0.0017

best p on a grid of 999 values: 0.700  (h/n = 0.7)


The same result holds for any data. The cell below flips a $p = 0.7$ coin 200
times, evaluates the log-likelihood $h \log p + (n-h)\log(1-p)$ at several
values of $p$, and finds its maximum on a grid. Setting the derivative
$h/p - (n-h)/(1-p)$ to zero gives $p = h/n$, so the grid maximum should equal
the observed frequency of heads. Minimizing the negative of this
log-likelihood is the cross-entropy loss used to train classifiers.

In [4]:
rng = np.random.default_rng(42)
true_p, n = 0.7, 200
flips = (rng.random(n) < true_p).astype(int)
h = int(flips.sum())
print(f"{n} flips of a p = {true_p} coin: {h} heads, h/n = {h / n:.4f}\n")

log_lik = lambda p: h * np.log(p) + (n - h) * np.log(1 - p)
print(f"{'p':>6s} {'log L(p)':>12s}")
for p in sorted([0.3, 0.5, 0.6, h / n, 0.8, 0.9]):
    marker = "   <- h/n" if p == h / n else ""
    print(f"{p:>6.3f} {log_lik(p):>12.4f}{marker}")

grid = np.linspace(0.001, 0.999, 999)
p_hat = grid[np.argmax(log_lik(grid))]
print(f"\ngrid argmax p_hat = {p_hat:.4f},  h/n = {h / n:.4f}")
assert abs(p_hat - h / n) < 1e-3, "MLE should equal empirical frequency"

200 flips of a p = 0.7 coin: 142 heads, h/n = 0.7100

     p     log L(p)
 0.300    -191.6513
 0.500    -138.6294
 0.600    -125.6821
 0.710    -120.4303   <- h/n
 0.800    -125.0338
 0.900    -148.5111

grid argmax p_hat = 0.7100,  h/n = 0.7100


## 3. The die and the $\sigma/\sqrt{n}$ error bar (§3.1, §3.6)

A fair die has mean $3.5$ and variance $35/12 \approx 2.92$, so its standard
deviation is $\sigma \approx 1.71$; the cell computes these exactly from the
six faces. The average of $n$ rolls has standard deviation $\sigma/\sqrt{n}$:
about 0.85 for 4 rolls and 0.17 for 100 rolls. The cell then averages the
first $n$ of 1,000,000 simulated rolls for growing $n$. Each error is one random
outcome, so it lands sometimes above and sometimes below the
$\sigma/\sqrt{n}$ column, but it shrinks along with that column: 100× more
rolls cut the typical error 10×.

In [5]:
faces = np.arange(1, 7)
mu = faces.mean()
var = np.mean((faces - mu) ** 2)
sigma = np.sqrt(var)
print(f"die: mean = {mu}, variance = {var:.4f} (35/12 = {35 / 12:.4f}), std = {sigma:.4f}")
print(f"standard error for 4 rolls = {sigma / 2:.2f}, for 100 rolls = {sigma / 10:.2f}\n")
assert abs(var - 35 / 12) < 1e-12

rng = np.random.default_rng(42)
rolls = rng.integers(1, 7, size=1_000_000)
print(f"{'n':>9s} {'average':>9s} {'|error|':>9s} {'sigma/sqrt(n)':>14s}")
for n in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    m = rolls[:n].mean()
    print(f"{n:>9,d} {m:>9.4f} {abs(m - mu):>9.4f} {sigma / np.sqrt(n):>14.4f}")

die: mean = 3.5, variance = 2.9167 (35/12 = 2.9167), std = 1.7078
standard error for 4 rolls = 0.85, for 100 rolls = 0.17

        n   average   |error|  sigma/sqrt(n)
       10    3.1000    0.4000         0.5401
      100    3.6400    0.1400         0.1708
    1,000    3.4880    0.0120         0.0540
   10,000    3.4752    0.0248         0.0171
  100,000    3.4998    0.0002         0.0054
1,000,000    3.4981    0.0019         0.0017


## 4. The central limit theorem (§3.6)

The cell takes the average of $n$ draws, repeats that 20,000 times, and
measures the standard deviation and the skewness of the 20,000 averages.
Skewness is the average cubed distance from the mean in units of the standard
deviation; it is 0 for any symmetric distribution and 2 for a single
exponential draw. Expected: the standard deviation of the averages matches
$\sigma/\sqrt{n}$, and the skewness of the averages falls toward 0 as $n$
grows, for the symmetric uniform source and for the lopsided exponential
source alike.

In [6]:
def skewness(x):
    """Third standardized moment: 0 for any symmetric distribution."""
    x = np.asarray(x, dtype=float)
    m, s = x.mean(), x.std()
    return np.mean(((x - m) / s) ** 3)


rng = np.random.default_rng(42)
n_trials = 20_000
sources = {
    # name: (sampler, mean, std, skewness of one draw)
    "Uniform(0, 1)": (lambda size: rng.random(size), 0.5, np.sqrt(1 / 12), 0.0),
    "Exponential(1)": (lambda size: rng.exponential(1.0, size=size), 1.0, 1.0, 2.0),
}

for name, (sampler, mu, sigma, skew1) in sources.items():
    print(f"Source: {name}  (mean = {mu}, std = {sigma:.4f}, skewness of one draw = {skew1:.1f})")
    print(f"  {'n':>5s} {'std of averages':>16s} {'sigma/sqrt(n)':>14s} {'skew of averages':>17s}")
    for n in [1, 5, 30, 100, 1000]:
        means = sampler((n_trials, n)).mean(axis=1)
        print(f"  {n:>5d} {means.std():>16.5f} {sigma / np.sqrt(n):>14.5f} "
              f"{skewness(means):>17.4f}")
    print()

print("Theory: the skewness of an average of n independent draws is (skewness of one draw)/sqrt(n),")
print(f"so the exponential at n = 100 should give about {2.0 / np.sqrt(100):.2f}.")

Source: Uniform(0, 1)  (mean = 0.5, std = 0.2887, skewness of one draw = 0.0)
      n  std of averages  sigma/sqrt(n)  skew of averages
      1          0.28858        0.28868           -0.0014
      5          0.12976        0.12910           -0.0137
     30          0.05227        0.05270            0.0035
    100          0.02861        0.02887           -0.0161
   1000          0.00916        0.00913           -0.0035

Source: Exponential(1)  (mean = 1.0, std = 1.0000, skewness of one draw = 2.0)
      n  std of averages  sigma/sqrt(n)  skew of averages
      1          1.00591        1.00000            2.0571
      5          0.44952        0.44721            0.9370
     30          0.18327        0.18257            0.3579


    100          0.10062        0.10000            0.1945


   1000          0.03158        0.03162            0.0623

Theory: the skewness of an average of n independent draws is (skewness of one draw)/sqrt(n),
so the exponential at n = 100 should give about 0.20.


## 5. Bessel's correction, by exact enumeration (§3.7)

Three rolls of a die have $6^3 = 216$ equally likely outcomes, so the average
of any estimator over all samples can be computed exactly, with no
simulation. The cell first reproduces the chapter's single sample 2, 6, 3:
the squared distances sum to 8.67 from $\bar x \approx 3.67$ and to 8.75 from
$\mu = 3.5$, and the difference is the gap $n(\bar x - \mu)^2 \approx 0.08$.
Then it averages both variance estimates over all 216 samples. Expected:
dividing by $n$ averages 1.94, which is $\tfrac{2}{3}$ of the true variance
2.92, and dividing by $n - 1$ averages exactly 2.92.

In [7]:
mu, true_var = 3.5, 35 / 12

x = np.array([2, 6, 3])
xbar = x.mean()
ss_sample = np.sum((x - xbar) ** 2)
ss_true = np.sum((x - mu) ** 2)
gap = len(x) * (xbar - mu) ** 2
print(f"rolls {x}: xbar = {xbar:.2f}")
print(f"  sum of squares from xbar = {ss_sample:.2f}, from mu = {ss_true:.2f}, "
      f"gap n(xbar - mu)^2 = {gap:.2f}")
assert abs(ss_sample + gap - ss_true) < 1e-12
print()

samples = np.array(list(itertools.product(range(1, 7), repeat=3)))  # all 216 samples
n = samples.shape[1]
ss = np.sum((samples - samples.mean(axis=1, keepdims=True)) ** 2, axis=1)
avg_div_n = np.mean(ss / n)
avg_div_n1 = np.mean(ss / (n - 1))
print(f"number of samples: {len(samples)}")
print(f"true variance:                    {true_var:.4f}")
print(f"average of divide-by-n estimate:  {avg_div_n:.4f}  ((n-1)/n * true = {(n - 1) / n * true_var:.4f})")
print(f"average of divide-by-(n-1):       {avg_div_n1:.4f}")
assert len(samples) == 216
assert abs(avg_div_n - (n - 1) / n * true_var) < 1e-12
assert abs(avg_div_n1 - true_var) < 1e-12

rolls [2 6 3]: xbar = 3.67
  sum of squares from xbar = 8.67, from mu = 8.75, gap n(xbar - mu)^2 = 0.08

number of samples: 216
true variance:                    2.9167
average of divide-by-n estimate:  1.9444  ((n-1)/n * true = 1.9444)
average of divide-by-(n-1):       2.9167
